# Stemming and Lemmatization of `xkcd-words.txt`

XKCD's [Simple Writer](https://xkcd.com/simplewriter/) (used for *Thing Explainer*) claims to only use the "ten hundred" (1,000) most common words. But `xkcd-words.txt`, scraped by `scripts/xkcdwords.py`, has 3,616 entries, because every inflected form (`add`, `added`, `adding`, `adds`, ...) is listed separately.

This notebook checks whether stemming or lemmatization can collapse those 3,616 word forms back down toward ~1,000 root words.

In [1]:
from collections import Counter, defaultdict

import nltk
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package wordnet to /Users/wmar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/wmar/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/wmar/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [2]:
words = [w for w in open('xkcd-words.txt').read().splitlines() if w]
len(words)

3616

## Stemming

Stemmers chop off suffixes using rules, without checking against a dictionary.

In [3]:
porter = PorterStemmer()
snowball = SnowballStemmer('english')

porter_stems = {w: porter.stem(w) for w in words}
snowball_stems = {w: snowball.stem(w) for w in words}

print(f'Porter:   {len(set(porter_stems.values()))} unique stems')
print(f'Snowball: {len(set(snowball_stems.values()))} unique stems')

Porter:   1496 unique stems
Snowball: 1492 unique stems


In [4]:
def group_words(mapping):
    groups = defaultdict(list)
    for word, key in mapping.items():
        groups[key].append(word)
    return groups

porter_groups = group_words(porter_stems)
print('Group sizes:', sorted(Counter(len(g) for g in porter_groups.values()).items()))

for group in sorted(porter_groups.values(), key=len, reverse=True)[:3]:
    print(len(group), sorted(group))

Group sizes: [(1, 569), (2, 386), (3, 60), (4, 343), (5, 118), (6, 13), (7, 5), (8, 1), (12, 1)]
12 ['travel', 'traveled', 'traveler', 'travelers', 'traveling', 'travelings', 'travelled', 'traveller', 'travellers', 'travelling', 'travellings', 'travels']
8 ['color', 'colored', 'colorer', 'colorers', 'colorful', 'coloring', 'colorings', 'colors']
7 ['breath', 'breathe', 'breathed', 'breathes', 'breathing', 'breathings', 'breaths']


Stemming alone gets us from 3,616 down to ~1,490 words — a big reduction, but still about 50% over the 1,000-word target.

## Lemmatization

Lemmatizers map words to dictionary headwords (lemmas), but need the part of speech to do it correctly. Naively assuming every word is a noun barely helps, since it doesn't touch verb forms like "added" or "adding":

In [5]:
lemmatizer = WordNetLemmatizer()

noun_lemmas = {w: lemmatizer.lemmatize(w, pos='n') for w in words}
print(f'Noun-only lemmas: {len(set(noun_lemmas.values()))} unique')

Noun-only lemmas: 2608 unique


Tagging each word's part of speech and lemmatizing accordingly does better, but POS tagging single words with no sentence context is unreliable:

In [6]:
def wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

tagged = nltk.pos_tag(words)
pos_lemmas = {w: lemmatizer.lemmatize(w, pos=wordnet_pos(tag)) for w, tag in tagged}
print(f'POS-tagged lemmas: {len(set(pos_lemmas.values()))} unique')

POS-tagged lemmas: 1729 unique


Instead, try lemmatizing each word as every part of speech and keep the shortest result. This catches irregular forms a stemmer would miss, like collapsing "am/are/is/was/were/being/been" toward "be":

In [7]:
def best_lemma(word):
    candidates = {lemmatizer.lemmatize(word, pos=p) for p in (wordnet.NOUN, wordnet.VERB, wordnet.ADJ, wordnet.ADV)}
    candidates.add(word)
    return min(candidates, key=lambda c: (len(c), c))

multi_lemmas = {w: best_lemma(w) for w in words}
print(f'Multi-POS lemmas: {len(set(multi_lemmas.values()))} unique')

{w: multi_lemmas[w] for w in ['am', 'are', 'is', 'was', 'were', 'be', 'been', 'being']}

Multi-POS lemmas: 1461 unique


{'am': 'am',
 'are': 'be',
 'is': 'be',
 'was': 'be',
 'were': 'be',
 'be': 'be',
 'been': 'be',
 'being': 'be'}

## Combining lemmatization with stemming

Lemmatizing first to normalize irregular forms, then stemming the result to merge regular derivations, combines both effects:

In [8]:
combo_porter = {w: porter.stem(multi_lemmas[w]) for w in words}
combo_snowball = {w: snowball.stem(multi_lemmas[w]) for w in words}

print(f'Lemma -> Porter:   {len(set(combo_porter.values()))} unique')
print(f'Lemma -> Snowball: {len(set(combo_snowball.values()))} unique')

combo_groups = group_words(combo_porter)
for group in sorted(combo_groups.values(), key=len, reverse=True)[:3]:
    print(len(group), sorted(group))

Lemma -> Porter:   1227 unique
Lemma -> Snowball: 1217 unique
12 ['travel', 'traveled', 'traveler', 'travelers', 'traveling', 'travelings', 'travelled', 'traveller', 'travellers', 'travelling', 'travellings', 'travels']
8 ['color', 'colored', 'colorer', 'colorers', 'colorful', 'coloring', 'colorings', 'colors']
7 ['are', 'be', 'been', 'being', 'is', 'was', 'were']


## Summary

In [9]:
results = {
    'Original word list': len(words),
    'Porter stem': len(set(porter_stems.values())),
    'Snowball stem': len(set(snowball_stems.values())),
    'WordNet lemma (noun only)': len(set(noun_lemmas.values())),
    'WordNet lemma (POS-tagged)': len(set(pos_lemmas.values())),
    'WordNet lemma (best of all POS)': len(set(multi_lemmas.values())),
    'Lemma + Porter stem': len(set(combo_porter.values())),
    'Lemma + Snowball stem': len(set(combo_snowball.values())),
    'Target': 1000,
}

for label, count in results.items():
    print(f'{count:>5}  {label}')

 3616  Original word list
 1496  Porter stem
 1492  Snowball stem
 2608  WordNet lemma (noun only)
 1729  WordNet lemma (POS-tagged)
 1461  WordNet lemma (best of all POS)
 1227  Lemma + Porter stem
 1217  Lemma + Snowball stem
 1000  Target


## Conclusion

Stemming and lemmatization both meaningfully shrink the word list, but neither gets close to 1,000 on its own:

- **Stemming** (Porter/Snowball) merges regular suffixes (`-ed`, `-ing`, `-s`) and gets to ~1,490 — about 50% over target.
- **Naive lemmatization** (assuming every word is a noun) barely helps, since it ignores verb inflections.
- **POS-tagged lemmatization** is held back by the unreliability of tagging single words out of context.
- **Lemmatizing then stemming** combines both effects and gets closest, to ~1,220 — still about 20% over target.

The "ten hundred words" in *Thing Explainer* refers to ~1,000 root words/concepts, but `xkcd-words.txt` enumerates *every* inflected form of those concepts (plurals, verb tenses, comparatives, etc.), so a 3,616 -> 1,000 collapse isn't a clean stemming/lemmatization problem. Closing the remaining gap would need manual curation — e.g. merging near-synonyms (`grey`/`gray`, `travel`/`travelled`/`traveled`) or dropping rarely-used derived forms (`allower`, `colorings`, `bagger`).